# 01 · Python en VS Code: solo lo que usa el proceso gaussiano

**Objetivo (45 min):** que puedas leer el notebook `02_proceso_gaussiano_desde_cero.ipynb` sin que la sintaxis te estorbe.
Cada ejemplo de aquí es una línea que aparece, tal cual, en ese notebook. No hay nada más.

Cómo trabajar en VS Code:

| Acción | Cómo |
|---|---|
| Ejecutar una celda | `Shift + Enter` (ejecuta y pasa a la siguiente) |
| Ver qué contiene una variable | escríbela sola al final de la celda, o `print(variable)` |
| Ayuda de una función | pasa el mouse encima, o escribe `np.linspace?` |
| Reiniciar si todo se enredó | botón *Restart* arriba del notebook y vuelve a ejecutar desde la primera celda |

## 1. Importar librerías y fijar el estilo

Las primeras líneas del notebook 02. `np` y `plt` son los apodos que usa todo el mundo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
AZUL, ORO, GRIS = "#003D79", "#D59F0F", "#8A8F98"   # colores; un texto entre comillas es un string
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})

## 2. Variables y números

En el notebook 02 los hiperparámetros del kernel se guardan en variables:

In [ ]:
ell     = 5.0      # días   (un decimal)
sigma_n = 2.5      # MW
n       = 200      # puntos de la malla (un entero)
dias    = 35
print(ell, sigma_n, n)
print(f"ℓ = {ell:.1f} días")        # f-string: texto con variables dentro; :.1f = un decimal

## 3. Arreglos de NumPy: la serie de demanda

Un arreglo es una lista de números que opera **toda a la vez**. Así se construye el eje de días y la serie sintética en el notebook 02.

In [ ]:
t = np.arange(dias, dtype=float)        # 0, 1, 2, ..., 34
print(t)
print("tamaño:", t.shape)

In [ ]:
tendencia = 120 + 25 / (1 + np.exp(-(t - 14) / 4))   # la fórmula se aplica a los 35 días de golpe
vaiven    = 5 * np.sin(2 * np.pi * t / 14)
print(np.round(tendencia, 1))

In [ ]:
rng   = np.random.default_rng(30)         # generador de números aleatorios con semilla: siempre los mismos
ruido = rng.normal(0, 2.5, dias)          # 35 números normales, media 0, desviación 2.5
y     = tendencia + vaiven + ruido        # sumar arreglos = sumar elemento a elemento
print(np.round(y, 1))

In [ ]:
print("promedio :", y.mean())
print("desviación:", y.std())
print("primer día:", y[0])          # Python cuenta desde 0
print("último día:", y[-1])
print("días 12 a 14:", y[12:15])    # del índice 12 al 14 (el 15 no entra)

## 4. Máscaras booleanas: qué días observamos

En el notebook 02 se esconden la próxima semana y tres días de medidor descompuesto. Se hace con un arreglo de `True`/`False`:

In [ ]:
observado = np.ones(len(t), dtype=bool)   # 35 valores True
observado[28:] = False                    # días 28..34: la próxima semana
observado[12:15] = False                  # días 12, 13, 14: medidor descompuesto
print(observado)

X, Y = t[observado], y[observado]         # solo los días True
print(f"{len(X)} días observados, {(~observado).sum()} sin dato")   # ~ invierte la máscara

## 5. Funciones de NumPy que aparecen en el notebook 02

In [ ]:
x = np.linspace(0, 34, n)     # n puntos igualmente espaciados entre 0 y 34: la malla donde predecimos
print(x[:5], "...", x[-1])
print(np.full(3, 7.5))          # arreglo lleno de un valor (media a priori b)
print(np.eye(3))                # matriz identidad (para sumar el ruido: sigma_n**2 * np.eye(N))
print(np.diag(np.eye(3) * 4))   # diagonal de una matriz (para la desviación: np.sqrt(np.diag(vpost)))
print(np.sqrt(np.array([4.0, 9.0])))

## 6. Funciones: el kernel

`def` empaqueta una receta. En el notebook 02 el kernel es una función de dos arreglos que devuelve una matriz:

In [ ]:
sigma_f = Y.std()

def k(a, c):
    """Kernel RBF entre todos los pares de días de `a` y de `c`. Devuelve una matriz len(a) x len(c)."""
    return sigma_f**2 * np.exp(-0.5 * (a[:, None] - c[None, :])**2 / ell**2)

K = k(X, X)
print("forma de k(X, X):", K.shape)      # 25 x 25
print("diagonal:", np.round(np.diag(K)[:3], 2), "= sigma_f² =", round(sigma_f**2, 2))

### Broadcasting: `a[:, None] - c[None, :]`

Es la línea más rara del kernel. `a[:, None]` convierte el vector en columna; al restarle un vector fila, NumPy expande ambos
y produce la **tabla de todas las diferencias** (fila i, columna j = a[i] − c[j]). Con tres números se ve claro:

In [ ]:
a = np.array([0., 1., 2.])
c = np.array([0., 5.])
print(a[:, None] - c[None, :])       # tabla 3 x 2

## 7. Matrices: `@`, `.T`, `cholesky` y `solve`

En NumPy `*` multiplica elemento por elemento y `@` es el **producto matricial**. `.T` transpone. Las seis líneas del notebook 02 usan esto:

In [ ]:
A = np.array([[4., 1.],
              [1., 3.]])
v = np.array([1., 2.])
print("A * 2 =\n", A * 2)          # elemento a elemento
print("A @ v =", A @ v)             # producto matricial: [4*1+1*2, 1*1+3*2]
print("A.T  =\n", A.T)

In [ ]:
L = np.linalg.cholesky(A)          # "raíz cuadrada" de la matriz: L @ L.T == A
print(np.round(L @ L.T, 10))

sol = np.linalg.solve(A, v)        # resuelve A @ sol = v  (nunca se invierte la matriz)
print("solución:", sol, " comprobación:", A @ sol)

Con estas piezas, así se ven las seis líneas del notebook 02 (no hace falta entenderlas hoy; solo reconocer que cada símbolo ya lo viste):

In [ ]:
N = len(X); b = Y.mean()
B2 = k(x, X)                                   # malla vs datos
B3 = k(X, X) + sigma_n**2 * np.eye(N)          # datos vs datos + ruido
R  = np.linalg.cholesky(B3).T
A_ = np.linalg.solve(R.T, B2.T).T
mpost = b + A_ @ np.linalg.solve(R.T, Y - b)   # media a posteriori
vpost = k(x, x) - A_ @ A_.T
stdpo = np.sqrt(np.diag(vpost))
print("media a posteriori:", mpost.shape, " desviación:", stdpo.shape)

## 8. Gráficas con matplotlib

Las cuatro órdenes que usa el notebook 02: `plot`, `fill_between` (la banda), `axvspan` (las franjas) y `legend`.

In [ ]:
plt.plot(X, Y, "o", color=AZUL, label="observado")             # "o" = puntos sin línea
plt.axvspan(27.5, 34.5, color=ORO, alpha=0.15, label="próxima semana (?)")   # alpha = transparencia
plt.axvspan(11.5, 14.5, color=GRIS, alpha=0.25, label="medidor descompuesto")
plt.xlabel("día"); plt.ylabel("demanda (MW)"); plt.legend(loc="upper left"); plt.title("Lo que sabemos")
plt.show()

In [ ]:
plt.fill_between(x, mpost - 2*stdpo, mpost + 2*stdpo, color=AZUL, alpha=0.15, label="banda 95 % (±2σ)")
plt.plot(x, mpost, color=AZUL, lw=2, label="media a posteriori")   # lw = grosor de línea
plt.plot(X, Y, "o", color=AZUL, label="datos")
plt.xlabel("día"); plt.ylabel("MW"); plt.legend(loc="upper left")
plt.show()

## 9. Un ciclo `for` sobre parejas

Aparece una sola vez en el notebook 02, para dibujar dos paneles con el mismo código:

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), sharey=True)      # dos paneles lado a lado
for a_, titulo in [(ax[0], "panel izquierdo"), (ax[1], "panel derecho")]:
    a_.plot(X, Y, "o", color=AZUL)
    a_.set_title(titulo); a_.set_xlabel("día")
ax[0].set_ylabel("MW")
plt.show()

## 10. Errores que verás (y qué hacer)

| Error | Causa típica | Arreglo |
|---|---|---|
| `NameError: name 'np' is not defined` | No ejecutaste la celda del `import` | Ejecuta las celdas en orden desde arriba |
| `IndexError: index 35 is out of bounds` | Pediste un elemento que no existe (recuerda: empieza en 0) | Revisa el índice |
| `ValueError: operands could not be broadcast` | Operaste arreglos de tamaños distintos | Imprime `.shape` de cada uno |
| `SyntaxError` con una flecha `^` | Falta un paréntesis o dos puntos | Mira justo donde apunta la flecha |
| `ModuleNotFoundError` | La librería no está instalada | En la terminal: `python -m pip install -r requirements.txt` |

---
**Listo.** Todo lo que usa `02_proceso_gaussiano_desde_cero.ipynb` ya lo viste: arreglos, máscaras, `linspace`/`exp`/`eye`/`diag`, funciones, broadcasting, `@`, `cholesky`/`solve` y `fill_between`.